# Experiment 8 - Per-Attribute Few-Shot LLM Prediction with Dual Evaluation
Half KB but easy version (no playing with KB and making it confusing)

## 1. Setup

In [2]:
import sys
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib64/python3.12/site-packages')
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib/python3.12/site-packages')
sys.path.append('/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI')

import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import pandas as pd
import numpy as np
import random
import time
import json
import re
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from sentence_transformers import SentenceTransformer, util

random.seed(42)
np.random.seed(42)

TARGET_ATTRIBUTES = [
    "bus_type", "model_number", "model",
    "read_speed_mb_s", "write_speed_mb_s",
    "height_mm", "width_mm"
]
NUMERIC_ATTRIBUTES = {"read_speed_mb_s", "write_speed_mb_s", "height_mm", "width_mm"}

print("Setup complete.")

Setup complete.


Set `USE_RAG = True` to enable retrieval from KB, `False` for LLM-only baseline.
Run twice — once with each setting — to get both configs for comparison.

In [17]:
USE_RAG = True  # Set to False for LLM-only, True for RAG

LLM_EXCEL = "results_exp8_per_attr_llm.csv"
RAG_EXCEL = "results_exp8_per_attr_rag.csv"
EXCEL_FILE = RAG_EXCEL if USE_RAG else LLM_EXCEL

## 2. Load Datasets

In [4]:
df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")
kb_full = pd.concat([df2, df3, df4], ignore_index=True)

print(f"Dataset 1: {len(df1)} rows")
print(f"KB full:   {len(kb_full)} rows")

Dataset 1: 812 rows
KB full:   2200 rows


## 3. Build Evaluation Set (25 query rows)

In [ ]:
def get_ground_truth(cluster_id, attribute):
    matches = kb_full[kb_full["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

MIN_PER_ATTR = 5  # minimum instances per attribute

# First pass: collect ALL recoverable missing tasks from df1
all_candidates = []
for idx, row in df1.iterrows():
    missing_attrs = []
    for attr in TARGET_ATTRIBUTES:
        if pd.isna(row.get(attr)):
            gt = get_ground_truth(row["cluster_id"], attr)
            if gt is not None:
                missing_attrs.append((attr, gt))
    if missing_attrs:
        all_candidates.append({"df1_idx": idx, "missing_attrs": missing_attrs})

# Second pass: greedily select rows to ensure MIN_PER_ATTR per attribute
attr_counts = {attr: 0 for attr in TARGET_ATTRIBUTES}
selected_rows = []
selected_idx = set()

# First: keep adding rows until every attribute has MIN_PER_ATTR instances
for candidate in all_candidates:
    if len(selected_rows) >= 25:
        break
    # Check if this row contributes to any under-represented attribute
    contributes = any(
        attr_counts[attr] < MIN_PER_ATTR
        for attr, _ in candidate["missing_attrs"]
    )
    if contributes:
        selected_rows.append(candidate)
        selected_idx.add(candidate["df1_idx"])
        for attr, _ in candidate["missing_attrs"]:
            attr_counts[attr] += 1

# Fill remaining slots up to 25 with any other rows
for candidate in all_candidates:
    if len(selected_rows) >= 25:
        break
    if candidate["df1_idx"] not in selected_idx:
        selected_rows.append(candidate)
        selected_idx.add(candidate["df1_idx"])
        for attr, _ in candidate["missing_attrs"]:
            attr_counts[attr] += 1

# Build eval dataframe
eval_records = []
for item in selected_rows:
    for attr, gt in item["missing_attrs"]:
        eval_records.append({
            "df1_idx": item["df1_idx"],
            "attribute": attr,
            "ground_truth": gt,
            "is_numeric": attr in NUMERIC_ATTRIBUTES
        })

eval_df = pd.DataFrame(eval_records)
query_indices = [item["df1_idx"] for item in selected_rows]
query_df = df1.loc[query_indices].copy()

print(f"Query rows:       {len(query_df)}")
print(f"Total eval tasks: {len(eval_df)}")
print()
print("Tasks per attribute (should all be >= 2):")
print(eval_df["attribute"].value_counts())

Query rows:       25
Total eval tasks: 51

Tasks per attribute (should all be >= 2):
attribute
model_number        11
read_speed_mb_s     10
bus_type             7
height_mm            7
width_mm             6
write_speed_mb_s     5
model                5
Name: count, dtype: int64


## 4. Build Knowledge Base (1000 rows = half of relevent kb + other)

In [7]:
query_cluster_ids = query_df["cluster_id"].tolist()
# relevant_kb = kb_full[kb_full["cluster_id"].isin(query_cluster_ids)].copy()
# relevant_kb = relevant_kb.sample(frac=0.5, random_state=42)
# remaining_needed = max(0, 1000 - len(relevant_kb))
# irrelevant_kb = kb_full[~kb_full["cluster_id"].isin(query_cluster_ids)]
# random_fill = irrelevant_kb.sample(n=min(remaining_needed, len(irrelevant_kb)), random_state=42)
# kb = pd.concat([relevant_kb, random_fill], ignore_index=True)
kb = kb_full
# print(f"Relevant KB rows (with GT): {len(relevant_kb)}")
# print(f"Random fill rows:           {len(random_fill)}")
print(f"Final KB size:              {len(kb)}")

Final KB size:              2200


## 5. Sanity Check

In [8]:
print("=== SANITY CHECK: Ground truth present in KB ===\n")
found = 0
not_found = 0
for _, task in eval_df.iterrows():
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]
    cluster_id = query_df.loc[idx, "cluster_id"]
    kb_matches = kb[kb["cluster_id"] == cluster_id]
    val_in_kb = any(
        pd.notna(r.get(attr)) and str(r.get(attr)).strip() == str(gt).strip()
        for _, r in kb_matches.iterrows()
    )
    status = "✓" if val_in_kb else "✗"
    found += int(val_in_kb)
    not_found += int(not val_in_kb)
    print(f"{status} Row {idx} | {attr:<20} | GT: {str(gt):<30} | KB matches: {len(kb_matches)}")

print(f"\nCoverage: {found}/{found+not_found} = {100*found/(found+not_found):.1f}%")

=== SANITY CHECK: Ground truth present in KB ===

✓ Row 0 | model_number         | GT: GV-N3080GAMING OC-10GD         | KB matches: 3
✓ Row 2 | model_number         | GT: CSSD-F960GBMP510               | KB matches: 3
✓ Row 2 | read_speed_mb_s      | GT: 3480.0                         | KB matches: 3
✓ Row 2 | write_speed_mb_s     | GT: 3000.0                         | KB matches: 3
✓ Row 3 | read_speed_mb_s      | GT: 226.0                          | KB matches: 3
✓ Row 4 | width_mm             | GT: 433.0                          | KB matches: 3
✓ Row 5 | bus_type             | GT: PCIe 3.0 x16                   | KB matches: 3
✓ Row 5 | model_number         | GT: 90YV0CV2-M0NA00                | KB matches: 3
✓ Row 6 | bus_type             | GT: USB 3.0                        | KB matches: 2
✓ Row 8 | model_number         | GT: GV-N166SOC-6GD                 | KB matches: 3
✓ Row 9 | bus_type             | GT: PCIe 3.0 x16                   | KB matches: 3
✓ Row 9 | model_number    

## 6. LLM Setup

In [9]:
predict_model = ChatOllama(model="llama3.1:8b", temperature=0)
eval_model = ChatOllama(model="llama3.1:8b", temperature=0)

test = predict_model.invoke("Say OK")
print("Ollama OK:", repr(test.content[:20]))

Ollama OK: 'OK'


## 7. Retriever Setup

In [18]:
LOCAL_MODEL_PATH = "/home/ma/ma_ma/ma_mpandya/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf"

print("Loading embedding model...")
embedding_model = SentenceTransformer(LOCAL_MODEL_PATH)

def row_to_text(row):
    attrs = ["title", "model", "model_number", "brand", "product_type"]
    return " | ".join([str(row[a]) for a in attrs if pd.notna(row.get(a))])

print("Encoding KB...")
kb_texts = kb.apply(row_to_text, axis=1).tolist()
kb_embeddings = embedding_model.encode(
    kb_texts, convert_to_tensor=True,
    batch_size=64, show_progress_bar=True
)
print(f"KB encoded: {len(kb_texts)} rows")

print("Encoding query rows...")
query_texts = query_df.apply(row_to_text, axis=1).tolist()
query_embeddings = embedding_model.encode(
    query_texts, convert_to_tensor=True,
    batch_size=64, show_progress_bar=True
)
print("Query rows encoded.")

# Map df1_idx to query embedding position
query_idx_to_pos = {idx: pos for pos, idx in enumerate(query_df.index)}

Loading embedding model...
Encoding KB...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

KB encoded: 2200 rows
Encoding query rows...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query rows encoded.


## 8. Per-Attribute Few-Shot Prompts

Each attribute has its own targeted prompt with 3 real examples from Dataset 2.

In [ ]:
FEW_SHOT_PROMPTS = {

"bus_type": """\
You are a product data expert. Extract ONLY the bus_type from the product text.
bus_type is the interface/connection type of the product (e.g. PCIe 3.0 x16, SATA III, USB 3.0).
Respond with VALUE:<answer> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ
VALUE:SATA III

Example 2:
Product: CORSAIR Force Series MP510 960GB M.2 SSD PCIe Gen3 x4, NVMe, up to 3480/3000MB/s
VALUE:PCIe 3.0 x4

Example 3:
Product: Kingston DataTraveler Vault Privacy 3.0 16GB. Description: USB 3.0 interface, hardware encryption
VALUE:USB 3.0

Now extract:
Product: {text}
VALUE:""",

"model_number": """\
You are a product data expert. Extract ONLY the model_number from the product text.
model_number is the exact manufacturer SKU or part number — usually alphanumeric with dashes, 
often found in parentheses or at the end of the product title.
Respond with VALUE:<answer> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ
VALUE:WD60EZAZ

Example 2:
Product: CORSAIR Force Series MP510 960GB M.2 SSD PCIe Gen3 x4 (CSSD-F960GBMP510). Description: PCI Express 3.0 x4
VALUE:CSSD-F960GBMP510

Example 3:
Product: Asus Dual GeForce GTX 1650 OC 4GB Graphics Card | 90YV0CV2-M0NA00. Description: 896 CUDA Cores
VALUE:90YV0CV2-M0NA00

Now extract:
Product: {text}
VALUE:""",

"model": """\
You are a product data expert. Extract ONLY the model name from the product text.
model is the product model name (not the full product title, not the SKU).
Examples of model names: WD Blue, Force Series MP510, Exos 7E8, GeForce RTX 3080 GAMING OC.
Respond with VALUE:<answer> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ
VALUE:WD Blue

Example 2:
Product: CORSAIR Force Series MP510 960GB M.2 SSD PCIe Gen3 x4, NVMe, up to 3480/3000MB/s
VALUE:Force Series MP510

Example 3:
Product: 4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/s, 512e, 128MB cache, 3.5-Inch HDD
VALUE:Exos 7E8

Now extract:
Product: {text}
VALUE:""",

"read_speed_mb_s": """\
You are a product data expert. Extract ONLY the sequential READ speed in MB/s from the product text.
Return the number only (e.g. 3480, 550, 3400). Do NOT return write speed.
Look for: "read", "R:", "read/write" (first number), "MB/s read", "MBps read".
Respond with VALUE:<number> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: CORSAIR Force Series MP510 960GB M.2 SSD, up to 3480/3000MB/s read/write
VALUE:3480

Example 2:
Product: Samsung 860 EVO 4TB 2.5" SATA SSD. Description: Read 550MB/s, Write 520MB/s
VALUE:550

Example 3:
Product: SSD 240GB 2.5'' PATRIOT Burst SATA3 R/W:555/500 MB/s 3D NAND
VALUE:555

Now extract:
Product: {text}
VALUE:""",

"write_speed_mb_s": """\
You are a product data expert. Extract ONLY the sequential WRITE speed in MB/s from the product text.
Return the number only (e.g. 3000, 520, 2500). Do NOT return read speed.
Look for: "write", "W:", "read/write" (second number), "MB/s write", "MBps write".
Respond with VALUE:<number> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: CORSAIR Force Series MP510 960GB M.2 SSD, up to 3480/3000MB/s read/write
VALUE:3000

Example 2:
Product: Samsung 860 EVO 4TB 2.5" SATA SSD. Description: Read 550MB/s, Write 520MB/s
VALUE:520

Example 3:
Product: SSD 240GB 2.5'' PATRIOT Burst SATA3 R/W:555/500 MB/s 3D NAND
VALUE:500

Now extract:
Product: {text}
VALUE:""",

"height_mm": """\
You are a product data expert. Extract ONLY the HEIGHT in millimeters from the product text.
Return the number only (e.g. 46.0, 10.5, 7.0). Height is typically the smallest dimension for drives.
Look for: "height", "H:", dimensions format HxWxD or HDW, "mm" measurements.
Do NOT confuse with width or length.
Respond with VALUE:<number> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: GeForce GTX 1660 Ti GAMING X 6G. Description: Dimensions: 46mm x 127mm x 247mm (H x W x D)
VALUE:46

Example 2:
Product: Samsung T5 1TB External SSD. Description: Dimensions (HDW): 10.5mm x 74mm x 57mm
VALUE:10.5

Example 3:
Product: Kingston DC500M 1.92TB 2.5" SSD. Description: SATA 3.0, 2.5", up to 555MB/s read, 7mm height
VALUE:7

Now extract:
Product: {text}
VALUE:""",

"width_mm": """\
You are a product data expert. Extract ONLY the WIDTH in millimeters from the product text.
Return the number only (e.g. 127.0, 57.0, 22.0). 
Look for: "width", "W:", dimensions format HxWxD or HDW, "mm" measurements.
Do NOT confuse with height or length.
Respond with VALUE:<number> only. If not found, respond with VALUE:UNKNOWN.

Example 1:
Product: GeForce GTX 1660 Ti GAMING X 6G. Description: Dimensions: 46mm x 127mm x 247mm (H x W x D)
VALUE:127

Example 2:
Product: Samsung T5 1TB External SSD. Description: Dimensions (HDW): 10.5mm x 74mm x 57mm
VALUE:57

Example 3:
Product: ADATA SU800 SATA M.2 2280 SSD 256GB. Description: 22x80x3.5mm dimensions
VALUE:22

Now extract:
Product: {text}
VALUE:"""
}

print("Per-attribute few-shot prompts defined for:", list(FEW_SHOT_PROMPTS.keys()))

Per-attribute few-shot prompts defined for: ['bus_type', 'model_number', 'model', 'read_speed_mb_s', 'write_speed_mb_s', 'height_mm', 'width_mm']


In [20]:
FEW_SHOT_PROMPTS_RAG = {}
for attr, prompt in FEW_SHOT_PROMPTS.items():
    rag_prompt = prompt.replace(
        "Now extract:\nProduct: {text}\nVALUE:",
        "Here are similar reference products from the knowledge base:\n{candidates}\n\nNow extract:\nProduct: {text}\nVALUE:"
    )
    FEW_SHOT_PROMPTS_RAG[attr] = rag_prompt

print("RAG prompts defined.")

RAG prompts defined.


## 9. Run Per-Attribute Prediction

In [21]:
def predict_attribute(product_text, attribute, predict_model, 
                      candidates=None):
    if candidates is not None:
        # Format candidate rows as text
        key_fields = ["title", "model", "model_number", 
                      "bus_type", "read_speed_mb_s", "write_speed_mb_s",
                      "height_mm", "width_mm"]
        cand_lines = []
        for _, c in candidates.iterrows():
            c_dict = {k: v for k, v in c.items()
                      if k in key_fields and pd.notna(c.get(k))}
            cand_lines.append(str(c_dict))
        cands_text = "\n".join(cand_lines)
        prompt_template = FEW_SHOT_PROMPTS_RAG[attribute]
        prompt = prompt_template.format(
            text=str(product_text)[:500],
            candidates=cands_text
        )
    else:
        prompt_template = FEW_SHOT_PROMPTS[attribute]
        prompt = prompt_template.format(text=str(product_text)[:600])

    response = predict_model.invoke([HumanMessage(content=prompt)])
    response_text = response.content.strip()
    for line in response_text.splitlines():
        line = line.strip()
        if line.upper().startswith("VALUE:"):
            value = line.split(":", 1)[1].strip().strip('"').strip("'")
            if value.upper() != "UNKNOWN" and value.lower() not in {"", "none", "nan", "null"}:
                return value
            return "UNKNOWN"
    cleaned = response_text.strip().strip('"').strip("'")
    if cleaned and len(cleaned) < 80 and "\n" not in cleaned:
        if cleaned.upper() not in {"UNKNOWN", "NONE", "NULL", "NAN", ""}:
            return cleaned
    return "UNKNOWN"

TOP_K = 3

print(f"Running per-attribute prediction (RAG={'ON' if USE_RAG else 'OFF'})...")

t0 = time.time()
total = len(eval_df)
predictions = []

for i, (_, task) in enumerate(eval_df.iterrows()):
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]
    text = query_df.loc[idx, "title_description"]

    # Retrieve top-k candidates if RAG enabled
    if USE_RAG:
        pos = query_idx_to_pos[idx]
        q_emb = query_embeddings[pos]
        scores = util.cos_sim(q_emb, kb_embeddings)[0]
        top_idx = np.argsort(-scores.cpu().numpy())[:TOP_K]
        candidates = kb.iloc[top_idx]
    else:
        candidates = None

    predicted = predict_attribute(text, attr, predict_model, candidates)

    elapsed = time.time() - t0
    eta = (elapsed / (i+1)) * (total - i - 1)
    print(f"  [{i+1}/{total}] Row {idx} | {attr:<20} | "
          f"GT: {str(gt):<25} | Pred: {predicted:<25} | ETA: {eta/60:.1f}min")

    predictions.append({
        "df1_idx": idx,
        "config": "RAG" if USE_RAG else "LLM-only",
        "attribute": attr,
        "is_numeric": task["is_numeric"],
        "ground_truth": gt,
        "predicted": predicted,
        "unknown": predicted == "UNKNOWN"
    })

results_df = pd.DataFrame(predictions)
print(f"\nDone in {time.time()-t0:.1f}s")

Running per-attribute prediction (RAG=ON)...
  [1/51] Row 0 | model_number         | GT: GV-N3080GAMING OC-10GD    | Pred: GV-N3080GAMING OC-10GD    | ETA: 5.8min
  [2/51] Row 2 | model_number         | GT: CSSD-F960GBMP510          | Pred: MP510                     | ETA: 3.9min
  [3/51] Row 2 | read_speed_mb_s      | GT: 3480.0                    | Pred: 3480                      | ETA: 4.0min
  [4/51] Row 2 | write_speed_mb_s     | GT: 3000.0                    | Pred: 3000                      | ETA: 4.0min
  [5/51] Row 3 | read_speed_mb_s      | GT: 226.0                     | Pred: UNKNOWN                   | ETA: 4.0min
  [6/51] Row 4 | width_mm             | GT: 433.0                     | Pred: UNKNOWN                   | ETA: 4.0min
  [7/51] Row 5 | bus_type             | GT: PCIe 3.0 x16              | Pred: PCIE 3.0                  | ETA: 3.9min
  [8/51] Row 5 | model_number         | GT: 90YV0CV2-M0NA00           | Pred: DUAL-OC-4G-G5             | ETA: 3.8min
  [9/51] Ro

## 10. Run Per-Attribute Prediction (LLM-only or RAG)

In [22]:
def normalize(val):
    return str(val).lower().strip()

def is_correct_standard(predicted, ground_truth, attribute):
    if predicted is None or str(predicted).strip().lower() in {
            "", "nan", "none", "unknown", "null"}:
        return False
    if attribute in NUMERIC_ATTRIBUTES:
        try:
            p = float(str(predicted).replace(",", "").strip())
            g = float(str(ground_truth).replace(",", "").strip())
            return abs(p - g) / abs(g) <= 0.10 if g != 0 else p == 0
        except:
            pass
    p = normalize(str(predicted))
    g = normalize(str(ground_truth))
    return p == g or p in g or g in p

results_df["correct_standard"] = results_df.apply(
    lambda row: is_correct_standard(row["predicted"], row["ground_truth"], row["attribute"]),
    axis=1
)

print("=" * 55)
print("STANDARD EVALUATION")
print("=" * 55)
print(f"Overall accuracy: {results_df['correct_standard'].mean():.3f}")
print(f"UNKNOWN rate:     {results_df['unknown'].mean():.3f}")
print(f"Total tasks:      {len(results_df)}")
print()
print("Per-attribute:")
print(results_df.groupby("attribute").agg(
    total=("correct_standard", "count"),
    correct=("correct_standard", "sum"),
    accuracy=("correct_standard", "mean"),
    unknown_rate=("unknown", "mean")
).round(3).to_string())
print()
print("Full prediction table:")
print(results_df[["attribute", "ground_truth", "predicted", "correct_standard"]]
      .to_string(index=False))

STANDARD EVALUATION
Overall accuracy: 0.529
UNKNOWN rate:     0.275
Total tasks:      51

Per-attribute:
                  total  correct  accuracy  unknown_rate
attribute                                               
bus_type              7        6     0.857         0.000
height_mm             7        5     0.714         0.286
model                 5        2     0.400         0.200
model_number         11        4     0.364         0.000
read_speed_mb_s      10        4     0.400         0.600
width_mm              6        2     0.333         0.667
write_speed_mb_s      5        4     0.800         0.200

Full prediction table:
       attribute           ground_truth                   predicted  correct_standard
    model_number GV-N3080GAMING OC-10GD      GV-N3080GAMING OC-10GD              True
    model_number       CSSD-F960GBMP510                       MP510              True
 read_speed_mb_s                 3480.0                        3480              True
write_speed_mb

## 11. LLM-Based Evaluation (per attribute, with KB context)

In [23]:
def llm_evaluate(predicted, ground_truth, attribute, cluster_id, kb, eval_model):
    """LLM judges prediction as correct/acceptable/wrong using GT + KB context."""
    if predicted == "UNKNOWN" or str(predicted).lower() in {"nan", "none", "null"}:
        return "wrong"

    # Get KB context rows for this cluster
    kb_matches = kb[kb["cluster_id"] == cluster_id]
    kb_context = ""
    for _, kb_row in kb_matches.head(3).iterrows():
        val = kb_row.get(attribute)
        title = kb_row.get("title", "")
        if pd.notna(val):
            kb_context += f"  - {str(title)[:80]} | {attribute}: {val}\n"

    eval_prompt = f"""You are evaluating a data cleaning prediction for a product database.

Attribute: {attribute}
Ground truth (from knowledge base): {ground_truth}
Predicted value: {predicted}

Knowledge base context (matching products):
{kb_context if kb_context else '  (none)'}

Judge as exactly one of:
- CORRECT: exact match or semantically equivalent
  (e.g. PCIe Gen3x4 = PCIe 3.0 x4, 3480 = 3480.0, SATA 6Gb/s = SATA III)
- ACCEPTABLE: partially correct, less/more detailed but not wrong
  (e.g. PCIe 3.0 when GT is PCIe 3.0 x16, or 550 when GT is 555)
- WRONG: clearly incorrect value

Respond with JUDGMENT:<label> only. Example: JUDGMENT:CORRECT"""

    response = eval_model.invoke([HumanMessage(content=eval_prompt)])
    response_text = response.content.strip().upper()

    if "JUDGMENT:CORRECT" in response_text:
        return "correct"
    elif "JUDGMENT:ACCEPTABLE" in response_text:
        return "acceptable"
    else:
        return "wrong"


print("Running LLM-based evaluation...")
llm_judgments = []
t0 = time.time()

for i, (_, row) in enumerate(results_df.iterrows()):
    cluster_id = query_df.loc[row["df1_idx"], "cluster_id"]
    judgment = llm_evaluate(
        row["predicted"], row["ground_truth"],
        row["attribute"], cluster_id, kb, eval_model
    )
    llm_judgments.append(judgment)
    print(f"  [{i+1}/{len(results_df)}] {row['attribute']:<20} | "
          f"GT: {str(row['ground_truth']):<25} | "
          f"Pred: {str(row['predicted']):<25} | {judgment}")

results_df["llm_judgment"] = llm_judgments

results_df.to_csv(EXCEL_FILE, index=False)

Running LLM-based evaluation...
  [1/51] model_number         | GT: GV-N3080GAMING OC-10GD    | Pred: GV-N3080GAMING OC-10GD    | correct
  [2/51] model_number         | GT: CSSD-F960GBMP510          | Pred: MP510                     | acceptable
  [3/51] read_speed_mb_s      | GT: 3480.0                    | Pred: 3480                      | correct
  [4/51] write_speed_mb_s     | GT: 3000.0                    | Pred: 3000                      | correct
  [5/51] read_speed_mb_s      | GT: 226.0                     | Pred: UNKNOWN                   | wrong
  [6/51] width_mm             | GT: 433.0                     | Pred: UNKNOWN                   | wrong
  [7/51] bus_type             | GT: PCIe 3.0 x16              | Pred: PCIE 3.0                  | acceptable
  [8/51] model_number         | GT: 90YV0CV2-M0NA00           | Pred: DUAL-OC-4G-G5             | correct
  [9/51] bus_type             | GT: USB 3.0                   | Pred: USB 3.0                   | correct
  [10/51] mo

## 12. Final Comparison: Standard vs LLM Evaluation

In [24]:
# Load both results if running separately, otherwise use results_df
# If you ran LLM-only first and saved it:
results_llm = pd.read_csv(LLM_EXCEL)
results_rag = pd.read_csv(RAG_EXCEL)
all_results = pd.concat([results_llm, results_rag], ignore_index=True)

# If both configs are already in results_df (ran sequentially):
all_results = results_df

print("=" * 70)
print("FINAL COMPARISON: LLM-only vs RAG — Standard & LLM Evaluation")
print("=" * 70)

for config in all_results["config"].unique():
    df_c = all_results[all_results["config"] == config]
    string_acc = df_c["correct_standard"].mean()
    llm_correct = (df_c["llm_judgment"] == "correct").mean()
    llm_acceptable = (df_c["llm_judgment"].isin(["correct", "acceptable"])).mean()
    print(f"\n--- {config} ---")
    print(f"  Standard accuracy:              {string_acc:.3f}")
    print(f"  LLM eval (correct only):        {llm_correct:.3f}")
    print(f"  LLM eval (correct+acceptable):  {llm_acceptable:.3f}")
    print(f"  UNKNOWN rate:                   {df_c['unknown'].mean():.3f}")

print("\n\nPer-attribute breakdown:")
per_attr = all_results.groupby(["config", "attribute"]).apply(lambda x: pd.Series({
    "total": len(x),
    "standard_acc": x["correct_standard"].mean(),
    "llm_correct": (x["llm_judgment"] == "correct").mean(),
    "llm_acceptable": (x["llm_judgment"].isin(["correct", "acceptable"])).mean(),
    "unknown_rate": x["unknown"].mean()
}), include_groups=False).round(3)
print(per_attr.to_string())

print("\nFull prediction table:")
print(all_results[["config", "attribute", "ground_truth", "predicted",
                    "correct_standard", "llm_judgment"]].to_string(index=False))

# Save combined results
all_results.to_csv("results_exp8_all.csv", index=False)
print("\n✓ Saved to results_exp8_all.csv")

FINAL COMPARISON: LLM-only vs RAG — Standard & LLM Evaluation

--- RAG ---
  Standard accuracy:              0.529
  LLM eval (correct only):        0.510
  LLM eval (correct+acceptable):  0.725
  UNKNOWN rate:                   0.275


Per-attribute breakdown:
                         total  standard_acc  llm_correct  llm_acceptable  unknown_rate
config attribute                                                                       
RAG    bus_type            7.0         0.857        0.571           1.000         0.000
       height_mm           7.0         0.714        0.714           0.714         0.286
       model               5.0         0.400        0.400           0.800         0.200
       model_number       11.0         0.364        0.455           1.000         0.000
       read_speed_mb_s    10.0         0.400        0.400           0.400         0.600
       width_mm            6.0         0.333        0.333           0.333         0.667
       write_speed_mb_s    5.0    